### 🎓 Student Dropout Prediction
#### Objective
Build machine learning models to predict student outcomes:
- Graduate
- Dropout
- Enrolled
#### Experiments
We compare two scenarios:
- Enrollment-only features
  → Can we predict dropout risk at the time of enrollment?
- Enrollment + First Semester features
  → How much does academic performance improve predictions?
#### Tools
- Databricks
- Delta Tables
- Pandas
- Scikit-learn
- MLflow (experiment tracking)

In [0]:
spark_df = spark.table("workspace.default.student_dropout_raw")
df = spark_df.toPandas()

df.head()

#### 📊Feature Set Definitions
We define two different feature sets to simulate different prediction timepoints:
- Enrollment-only features: Available at admission time.
- First semester features: Includes academic performance from semester 1.

This allows us to measure how predictive power changes over time.

In [0]:
enrollment_only_cols = [
    col for col in df.columns 
    if not col.startswith("curricular_units_1st_sem")
    and not col.startswith("curricular_units_2nd_sem")
    and col != "target"
]

first_sem_cols = [
    col for col in df.columns
    if not col.startswith("curricular_units_2nd_sem")
    and col != "target"
]

#### 🎯 Target Encoding

The target variable (Graduate, Dropout, Enrolled) is label-encoded into integers for modeling.

In [0]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["target_encoded"] = le.fit_transform(df["target"])

In [0]:
dict(zip(le.classes_, le.transform(le.classes_)))

#### 🔀 Train/Test Split

We use a stratified 80/20 split to preserve class distribution across training and testing sets.

In [0]:
from sklearn.model_selection import train_test_split

X1 = df[enrollment_only_cols]
X2 = df[first_sem_cols]
y = df["target_encoded"]

X1_train, X1_test, y_train, y_test = train_test_split(
    X1, y, test_size=0.2, stratify=y, random_state=42
)

X2_train, X2_test, _, _ = train_test_split(
    X2, y, test_size=0.2, stratify=y, random_state=42
)

#### 🧩 Feature Typing

We separate:
- Numerical features → StandardScaler
- Categorical features → OneHotEncoder

This preprocessing will be handled inside a Scikit-learn Pipeline to prevent data leakage.

In [0]:
numeric_cols = [
    "previous_qualification_grade",
    "admission_grade",
    "age_at_enrollment",
    "curricular_units_1st_sem_grade",
    "curricular_units_2nd_sem_grade",
    "unemployment_rate",
    "inflation_rate",
    "gdp",
]

In [0]:
all_features = [col for col in df.columns if col not in ["target", "target_encoded"]]

categorical_cols = [col for col in all_features if col not in numeric_cols]

In [0]:
# For X1 (Enrollment-only features)
numeric_cols_X1 = [c for c in numeric_cols if c in X1_train.columns]
categorical_cols_X1 = [c for c in categorical_cols if c in X1_train.columns]

# For X2 (Enrollment + 1st sem features)
numeric_cols_X2 = [c for c in numeric_cols if c in X2_train.columns]
categorical_cols_X2 = [c for c in categorical_cols if c in X2_train.columns]

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

In [0]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.ensemble import RandomForestClassifier

# log_reg = LogisticRegression(
#     max_iter=1000,
#     class_weight="balanced"
# )

# rf = RandomForestClassifier(
#     n_estimators=200,
#     class_weight="balanced",
#     random_state=42
# )

In [0]:
# log_pipeline = Pipeline(
#     steps=[
#         ("preprocessor", preprocessor),
#         ("classifier", log_reg),
#     ]
# )

# rf_pipeline = Pipeline(
#     steps=[
#         ("preprocessor", preprocessor),
#         ("classifier", rf),
#     ]
# )

#### 📈 Experiment Tracking with MLflow

All models are logged to MLflow, including:
- Model parameters
- Accuracy
- Macro F1 score
- Full pipeline (preprocessing + model)
- Registered model version
This ensures reproducibility and experiment comparison.

In [0]:
import mlflow

mlflow.set_experiment("/Shared/student_dropout_prediction")

In [0]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def train_and_log(model_pipeline, model_name, X_train, X_test, y_train, y_test, feature_set):
    
    with mlflow.start_run(run_name=model_name):
        
        model_pipeline.fit(X_train, y_train)
        preds = model_pipeline.predict(X_test)
        
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average="macro")
        
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("feature_set", feature_set)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("macro_f1", f1)
        
        mlflow.set_tag("problem_type", "multiclass_classification")
        mlflow.set_tag("class_imbalance", "true")
        
        mlflow.sklearn.log_model(
            model_pipeline,
            "model",
            registered_model_name=f"StudentDropout_{model_name}",
            input_example=X_train.head(3)
        )
        
        print(f"{model_name} Accuracy:", acc)
        print(f"{model_name} Macro F1:", f1)
        print("\nClassification Report:\n")
        print(classification_report(y_test, preds))

        return {
            "model": model_name,
            "feature_set": feature_set,
            "accuracy": acc,
            "macro_f1": f1
        }

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def make_pipeline(numeric_cols, categorical_cols, model_type="logistic"):
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ]
    )
    
    if model_type=="logistic":
        classifier = LogisticRegression(max_iter=1000, class_weight="balanced")
    else:
        classifier = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
    
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])
    
    return pipeline

# Pipelines for X1
log_pipeline_X1 = make_pipeline(numeric_cols_X1, categorical_cols_X1, "logistic")
rf_pipeline_X1 = make_pipeline(numeric_cols_X1, categorical_cols_X1, "rf")

# Pipelines for X2
log_pipeline_X2 = make_pipeline(numeric_cols_X2, categorical_cols_X2, "logistic")
rf_pipeline_X2 = make_pipeline(numeric_cols_X2, categorical_cols_X2, "rf")

####🧪 Model Experiments

We train four models:

#####Enrollment-only Feature Set
- Logistic Regression
- Random Forest

#####Enrollment + First Semester Feature Set
- Logistic Regression
- Random Forest

In [0]:
results = []
# Model 1: Enrollment only
results.append(
    train_and_log(log_pipeline_X1, "Logistic_Enrollment",
                  X1_train, X1_test, y_train, y_test, "Enrollment")
)

results.append(
    train_and_log(rf_pipeline_X1, "RandomForest_Enrollment",
                  X1_train, X1_test, y_train, y_test, "Enrollment")
)
# Model 2: Enrollment + 1st semester
results.append(
    train_and_log(log_pipeline_X2, "Logistic_FirstSemester",
                  X2_train, X2_test, y_train, y_test, "FirstSemester")
)

results.append(
    train_and_log(rf_pipeline_X2, "RandomForest_FirstSemester",
                  X2_train, X2_test, y_train, y_test, "FirstSemester")
)

####📊 Results Summary
#####Enrollment-only models
- Moderate predictive performance.
- Dropout recall is limited.
- Early prediction is possible but uncertain.

#####Enrollment + First Semester models

- Significant performance improvement.
- Academic performance strongly influences outcome prediction.

#####Key Insight
Student performance during the first semester provides substantial predictive signal. Early intervention strategies may benefit from monitoring first-semester metrics.

In [0]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def plot_confusion_matrix(model, X_test, y_test, title):
    
    preds = model.predict(X_test)
    cm = confusion_matrix(y_test, preds)
    
    plt.figure()
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=["Dropout", "Enrolled", "Graduate"],
        yticklabels=["Dropout", "Enrolled", "Graduate"]
    )
    
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

####🔎 Class-Level Performance Analysis (Confusion Matrices)
While accuracy and macro F1 provide overall performance metrics, they do not reveal which classes are being misclassified.

Since the primary business objective is identifying students at risk of dropout, it is important to evaluate:
- How well the model detects Dropout cases (recall)
- Whether the model confuses Enrolled and Dropout
- Whether predictions are biased toward the majority class (Graduate)

The confusion matrices below allow us to inspect class-level behavior and identify potential weaknesses in the models.

In [0]:
plot_confusion_matrix(
    log_pipeline_X2,
    X2_test,
    y_test,
    "Confusion Matrix – Logistic Regression (First Semester Features)"
)

In [0]:
plot_confusion_matrix(
    rf_pipeline_X2,
    X2_test,
    y_test,
    "Confusion Matrix – Random Forest (First Semester Features)"
)

####📊 Interpretation
#####Logistic Regression (First Semester Features)
- More balanced predictions across classes.
- Dropout recall is moderate, meaning some at-risk students are successfully identified.
- Some confusion remains between Enrolled and Dropout.

#####Random Forest (First Semester Features)
- Strong performance on Graduate class.
- Lower recall for Dropout class, indicating difficulty detecting minority cases.
- Tendency to favor the majority class.

#####Key Insight

Although Random Forest achieves slightly higher overall accuracy, Logistic Regression provides more balanced class performance.

For an early-warning intervention system, balanced detection of Dropout cases may be more valuable than maximizing overall accuracy.

In [0]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df

####📊 Model Performance Comparison
We compare accuracy and macro F1 score across all models and feature sets to evaluate:
- The impact of first semester academic data
- Differences between Logistic Regression and Random Forest
- Trade-offs between overall accuracy and class balance

In [0]:
import matplotlib.pyplot as plt

results_df.plot(
    kind="bar",
    x="model",
    y=["accuracy", "macro_f1"]
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
import pandas as pd

feature_names = rf_pipeline_X2.named_steps["preprocessor"].get_feature_names_out()
importances = rf_pipeline_X2.named_steps["classifier"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False).head(15)

importance_df.plot(kind="barh", x="feature", y="importance")
plt.gca().invert_yaxis()
plt.show()

In [0]:
df["target"].value_counts().plot(kind="bar")
plt.title("Class Distribution")
plt.show()